# Section 5.4 — Scaling Limits of the SP Heuristic

Reproduces **Table 7** from the thesis.  
Runs the **SP heuristic (B-PHA)** on a synthetic 70-cage instance.

The 70-cage fleet is constructed by replicating the Loc1 12-cage pattern
across 5 additional synthetic locations (empty start) plus a partial 6th
location of 10 cages — giving 12 + 4×12 + 10 = 70 cages total.
Only Loc1 retains its real initial stock; the synthetic locations start empty.
DE and EEV are not attempted — DE is out-of-memory at this scale.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'models'))

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df as full_units_df,
    temps_bad_12, temps_normal_12, temps_good_12,
)
from SP import AugmentedLagrangianDecomposition


In [ ]:
# Fleet builder

def build_synthetic_fleet(n_target):
    """
    Build a synthetic n_target-cage fleet by replicating the Loc1 12-cage
    pattern across as many locations as needed.
    Only Loc1 keeps its original initial stock; synthetic locations start empty.
    Each location is assigned the same MAB as Loc1 (4 000 000 kg).
    """
    base = full_units_df.copy()  # 12 rows in Loc1
    n_base = len(base)

    rows = []
    loc_num = 1
    remaining = n_target

    while remaining > 0:
        take = min(n_base, remaining)
        chunk = base.iloc[:take].copy()
        chunk['location'] = f'Loc{loc_num}'
        if loc_num > 1:
            chunk['count'] = float('nan')
            chunk['avg_weight_g'] = float('nan')
        rows.append(chunk)
        remaining -= take
        loc_num += 1

    df = pd.concat(rows, ignore_index=True)

    loc_mab_multi = {
        f'Location {i + 1}': loc_mab['Location 1']
        for i in range(loc_num - 1)
    }

    return df, loc_mab_multi


units_df_70, loc_mab_70 = build_synthetic_fleet(70)
print(f'Fleet size: {len(units_df_70)} cages across {units_df_70["location"].nunique()} locations')
print(units_df_70.groupby('location').size().to_frame('cages'))


In [ ]:
# SP heuristic on 70-cage fleet  (mip_gap=2%, matches thesis Table 7)

MIP_GAP = 0.02

print('Building SP model (70 cages)...')
ald = AugmentedLagrangianDecomposition(
    units_df=units_df_70,
    loc_mab=loc_mab_70,
    regional_mab=regional_mab,
    T=T,
    mip_gap=MIP_GAP,
    temps_bad=temps_bad_12,
    temps_normal=temps_normal_12,
    temps_good=temps_good_12,
)
ald.build()
ald.solve()


In [ ]:
# Results — Table 7

print('\n' + '='*50)
print(f'Scenarios       : {ald.n_feasible}/{ald.n_scenarios} feasible')
print(f'E[obj] (MNOK)   : {ald.eval_obj / 1e6:.1f}')
print(f'MIP gap (%)     : {MIP_GAP * 100:.0f}')
print(f'Wall clock (s)  : {ald.total_time:.1f}')
print('='*50)

df_result = pd.DataFrame([{
    'Metric': 'Scenarios',
    'SP': f'{ald.n_feasible}/{ald.n_scenarios} feasible',
}, {
    'Metric': 'E[obj] (MNOK)',
    'SP': round(ald.eval_obj / 1e6, 1),
}, {
    'Metric': 'MIP gap tolerance (%)',
    'SP': MIP_GAP * 100,
}, {
    'Metric': 'Wall clock time (s)',
    'SP': round(ald.total_time, 1),
}]).set_index('Metric')

display(df_result)
